# UCS420: Cognitive Computing — Assignment 6

## NumPy — II

Name: Vishu Verma
Roll number: 1024170242

In [1]:
import numpy as np

## Question 1

Sensor readings with an average measurement error of +2°C. Everything below is done without a `for` loop.

In [2]:
temperature = np.array([25, 28, 31, 35, 38, 27, 33, 40])
print("Raw readings:", temperature)

Raw readings: [25 28 31 35 38 27 33 40]


**a) Add 2°C to every reading using vectorization.**

In [3]:
corrected = temperature + 2   # the scalar 2 is broadcast to all 8 readings
print("Corrected readings:", corrected)

Corrected readings: [27 30 33 37 40 29 35 42]


**b) Convert all temperatures from Celsius to Fahrenheit using $F = \frac{9}{5}C + 32$.**

In [4]:
fahrenheit = corrected * 9 / 5 + 32
print("Celsius   :", corrected)
print("Fahrenheit:", fahrenheit)

Celsius   : [27 30 33 37 40 29 35 42]
Fahrenheit: [ 80.6  86.   91.4  98.6 104.   84.2  95.  107.6]


**c) Identify all readings greater than 32°C using Boolean indexing.**

The corrected readings are used from here on, since they are the true temperatures.

In [5]:
mask = corrected > 32
print("Boolean mask      :", mask)
print("Readings above 32 :", corrected[mask])

# the same check on the raw (uncorrected) readings, for comparison
print("\nRaw readings above 32      :", temperature[temperature > 32])

Boolean mask      : [False False  True  True  True False  True  True]
Readings above 32 : [33 37 40 35 42]

Raw readings above 32      : [35 38 33 40]


**d) Count how many readings exceed 32°C.**

In [6]:
print("Readings above 32°C:", np.sum(mask))   # True counts as 1, False as 0
print("Same with count_nonzero:", np.count_nonzero(mask))
print("Raw readings above 32°C (before the +2 correction):", np.sum(temperature > 32))

Readings above 32°C: 5
Same with count_nonzero: 5
Raw readings above 32°C (before the +2 correction): 4


**e) How vectorization and Boolean indexing make data processing more efficient.**

- **No Python loop.** `temperature + 2` runs one compiled C loop over a contiguous block of memory.
  A Python `for` loop has to fetch each value as a Python object, check its type and build a new
  object for the result, once per element. For large arrays the vectorized form is often 10–100× faster.
- **Less code, fewer bugs.** `corrected[corrected > 32]` says what is wanted in one line. There is no
  index variable, no empty list to append to and no off-by-one mistakes.
- **The mask can be reused.** The Boolean array is built once and then used to select, count
  (`mask.sum()`) or update values (`corrected[mask] = ...`) without scanning the data again by hand.
- **It scales.** The same expression works unchanged for 8 readings or 8 million, and for 2-D data too.

In [7]:
# a quick timing comparison on one million readings
import time

big = np.random.default_rng(0).integers(20, 45, size=1_000_000)

t0 = time.perf_counter()
loop_result = [x for x in big if x + 2 > 32]
t1 = time.perf_counter()
vec_result = big[big + 2 > 32]
t2 = time.perf_counter()

print(f"Python loop : {(t1 - t0) * 1000:7.1f} ms")
print(f"Vectorized  : {(t2 - t1) * 1000:7.1f} ms")
print("Same answer :", len(loop_result) == len(vec_result))

Python loop :    59.9 ms
Vectorized  :     4.6 ms
Same answer : True


## Question 2

Daily steps for four users (rows) over three days (columns).

In [8]:
steps = np.array([
    [5000, 6200, 7100],
    [8000, 7500, 9000],
    [4500, 5100, 4800],
    [9000, 8500, 9500]])
print(steps)
print("Shape:", steps.shape, "→ 4 users × 3 days")

[[5000 6200 7100]
 [8000 7500 9000]
 [4500 5100 4800]
 [9000 8500 9500]]
Shape: (4, 3) → 4 users × 3 days


In [9]:
print("a) Total steps recorded :", steps.sum())
print("b) Mean number of steps :", steps.mean())
print("c) Maximum value        :", steps.max())
print("   Minimum value        :", steps.min())

a) Total steps recorded : 84200
b) Mean number of steps : 7016.666666666667
c) Maximum value        : 9500
   Minimum value        : 4500


In [10]:
# axis=0 collapses the rows, so one total per column (day)
print("d) Total steps per day  :", steps.sum(axis=0))
# axis=1 collapses the columns, so one total per row (user)
print("e) Total steps per user :", steps.sum(axis=1))

d) Total steps per day  : [26500 27300 30400]
e) Total steps per user : [18300 24500 14400 27000]


In [11]:
flat_index = steps.argmax()                            # index into the flattened array
user, day = np.unravel_index(flat_index, steps.shape)  # turn it back into (row, column)
print("f) argmax() on the flattened array:", flat_index)
print(f"   Position (user, day) = ({user}, {day}) → User {user + 1}, Day {day + 1}, value {steps[user, day]}")

f) argmax() on the flattened array: 11
   Position (user, day) = (3, 2) → User 4, Day 3, value 9500


## Question 3

**a) Create the array.**

In [12]:
original = np.array([1, 2, 3, 4, 5, 6])
print("original:", original)

original: [1 2 3 4 5 6]


**b) Slice from index 1 to 4 and store it in `subset`.**

In [13]:
subset = original[1:5]   # indices 1, 2, 3, 4 (the stop index is excluded)
print("subset:", subset)

subset: [2 3 4 5]


**c) Change the first element of `subset` to 999.**

In [14]:
subset[0] = 999
print("subset  :", subset)
print("original:", original)
print("Shares memory:", np.shares_memory(original, subset))

subset  : [999   3   4   5]
original: [  1 999   3   4   5   6]
Shares memory: True


A basic slice is a **view**: `subset` points into the same memory as `original`, so changing
`subset[0]` also changed `original[1]`.

**d) Slice again, this time with `.copy()`, and change its first element to 500.**

In [15]:
copied = original[1:5].copy()
copied[0] = 500
print("copied  :", copied)
print("original:", original)   # unchanged by this step
print("Shares memory:", np.shares_memory(original, copied))

copied  : [500   3   4   5]
original: [  1 999   3   4   5   6]
Shares memory: False


`.copy()` allocates new memory, so the change stays in `copied` and `original` is not touched.

**e) Numbers 1 to 12 with `np.arange()`, reshaped to 3 × 4.**

In [16]:
matrix = np.arange(1, 13).reshape(3, 4)
print(matrix)

[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]


**f) Indexing and slicing.**

In [17]:
print("First row :", matrix[0])
print("Last row  :", matrix[-1])
print("Second column:", matrix[:, 1])
print("Rows 1–2, columns 2–3:")
print(matrix[1:3, 2:4])

First row : [1 2 3 4]
Last row  : [ 9 10 11 12]
Second column: [ 2  6 10]
Rows 1–2, columns 2–3:
[[ 7  8]
 [11 12]]


Rows 1–2 and columns 2–3 are read as 0-based indices here (`matrix[1:3, 2:4]`).
If they are meant as 1-based positions (first and second row, second and third column) the slice is:

In [18]:
print(matrix[0:2, 1:3])

[[2 3]
 [6 7]]


**g) Flatten with both `flatten()` and `ravel()`.**

In [19]:
flat_f = matrix.flatten()
flat_r = matrix.ravel()
print("flatten():", flat_f)
print("ravel()  :", flat_r)

flatten(): [ 1  2  3  4  5  6  7  8  9 10 11 12]
ravel()  : [ 1  2  3  4  5  6  7  8  9 10 11 12]


**h) Modify the array returned by `ravel()`.**

In [20]:
flat_r[0] = 100
print("ravel() array:", flat_r)
print("matrix after changing the ravel() array:")
print(matrix)

ravel() array: [100   2   3   4   5   6   7   8   9  10  11  12]
matrix after changing the ravel() array:
[[100   2   3   4]
 [  5   6   7   8]
 [  9  10  11  12]]


`matrix[0, 0]` is now 100. `ravel()` returns a **view** whenever it can (the matrix is contiguous here),
so writing to it writes to the matrix.

**i) Modify the array returned by `flatten()`.**

In [21]:
flat_f[1] = 200
print("flatten() array:", flat_f)
print("matrix after changing the flatten() array:")
print(matrix)

flatten() array: [  1 200   3   4   5   6   7   8   9  10  11  12]
matrix after changing the flatten() array:
[[100   2   3   4]
 [  5   6   7   8]
 [  9  10  11  12]]


`matrix[0, 1]` is still 2. `flatten()` always returns a **copy**, so the matrix is not affected.

**j) Attributes of the 3 × 4 matrix.**

In [22]:
print("shape:", matrix.shape)
print("ndim :", matrix.ndim)
print("size :", matrix.size)
print("dtype:", matrix.dtype)

shape: (3, 4)
ndim : 2
size : 12
dtype: int64


## Question 4

Predict a user's assistance score from sleep hours, activity level and stress level.
Each row of `X` is one user, each column is one feature.

In [23]:
X = np.array([
    [6, 70, 3],
    [5, 50, 6],
    [8, 80, 2],
    [4, 30, 8]
])
y = np.array([40, 65, 30, 85])

**a) Shape and dimensions of X.**

In [24]:
print("Shape:", X.shape, "→ 4 users, 3 features")
print("ndim :", X.ndim)

Shape: (4, 3) → 4 users, 3 features
ndim : 2


**b) Transpose of X.**

In [25]:
print(X.T)
print('Shape:', X.T.shape)

[[ 6  5  8  4]
 [70 50 80 30]
 [ 3  6  2  8]]
Shape: (3, 4)


The transpose swaps rows and columns. In `X` each row is a **user**; in `X.T` each row is a **feature**:
row 0 holds every user's sleep hours, row 1 every activity level and row 2 every stress level.
This is the shape needed to multiply features against users in the normal equation.

**c) Matrix product `X.T @ X`.**

In [26]:
XtX = X.T @ X
print(XtX)
print("Shape:", XtX.shape)

[[  141  1430    96]
 [ 1430 14700   910]
 [   96   910   113]]
Shape: (3, 3)


The result is 3 × 3 and symmetric. The diagonal is the sum of squares of each feature and the other
entries are the sums of products between pairs of features.

**d) Matrix inverse with `np.linalg.inv()`.**

In [27]:
# an inverse only exists for a square matrix with a non-zero determinant
print("Determinant of X.T @ X:", np.linalg.det(XtX))
print("Rank:", np.linalg.matrix_rank(XtX), "of", XtX.shape[0])

XtX_inv = np.linalg.inv(XtX)
print("\nInverse:")
print(XtX_inv)
print("\nCheck (X.T @ X) @ inverse ≈ identity:")
print(np.round(XtX @ XtX_inv, 6) + 0.0)   # + 0.0 turns -0. into 0.

Determinant of X.T @ X: 753700.0000000029
Rank: 3 of 3

Inverse:
[[ 1.10521428 -0.09848746 -0.14581398]
 [-0.09848746  0.00891203  0.01190129]
 [-0.14581398  0.01190129  0.0368847 ]]

Check (X.T @ X) @ inverse ≈ identity:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


`X` itself is 4 × 3 and not square, so it has no inverse. `X.T @ X` is square and its
determinant is not zero, so `np.linalg.inv()` is appropriate here.

**e) Ordinary Least Squares: $\beta = (X^TX)^{-1}X^Ty$.**

In [28]:
beta = XtX_inv @ X.T @ y
print("β =", beta)

# cross-check with NumPy's least-squares solver
beta_check, *_ = np.linalg.lstsq(X, y, rcond=None)
print("lstsq gives:", beta_check)
print("Match:", np.allclose(beta, beta_check))

β = [-0.15921454  0.14083853 10.06302242]
lstsq gives: [-0.15921454  0.14083853 10.06302242]
Match: True


In [29]:
y_hat = X @ beta
print("Actual    :", y)
print("Predicted :", np.round(y_hat, 2))
print("Residuals :", np.round(y - y_hat, 2))

Actual    : [40 65 30 85]
Predicted : [39.09 66.62 30.12 84.09]
Residuals : [ 0.91 -1.62 -0.12  0.91]


**f) What the coefficients mean.**

In [30]:
for name, b in zip(["Sleep hours", "Activity level", "Stress level"], beta):
    print(f"{name:15s}: {b:+.4f}")

Sleep hours    : -0.1592
Activity level : +0.1408
Stress level   : +10.0630


The model is `score = β₁·sleep + β₂·activity + β₃·stress`. Each coefficient is the change in the predicted
assistance score when that feature goes up by one unit and the other two stay the same.

- **Sleep hours (β₁ ≈ −0.16):** one extra hour of sleep lowers the predicted score by about 0.16 points
  when activity and stress stay the same. More sleep means slightly less assistance, but the effect is tiny.
- **Activity level (β₂ ≈ +0.14):** one extra unit of activity adds about 0.14 points. On its own, higher
  activity goes with *lower* scores in this data, but activity and stress move closely together (active
  users are less stressed). The stress coefficient already explains that drop, so activity is left with
  only a small positive correction.
- **Stress level (β₃ ≈ +10.06):** one extra point of stress adds about 10 points to the score. This is by
  far the strongest factor: in this model, stress drives how much assistance a user needs.

There is no intercept column, so the model is forced through the origin. Only four users are used to fit
three coefficients, and the features are strongly correlated with each other. The fit is close
(predictions are within about 1.6 points of the actual scores), but the coefficients describe this
small sample, not a general rule.

**g) Predict the score for a new user.**

In [31]:
new_user = np.array([5, 40, 7])
prediction = new_user @ beta
print("New user [sleep, activity, stress]:", new_user)
print(f"Predicted assistance score: {prediction:.2f}")

New user [sleep, activity, stress]: [ 5 40  7]
Predicted assistance score: 75.28


The predicted score is about 75.3. With a stress level of 7, this user falls between user 2
(stress 6, score 65) and user 4 (stress 8, score 85), so they need a high level of assistance.